# Lesson 8 · Special relativity

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/08_special_relativity.ipynb)

**For:** IB Physics (A.5 Galilean and special relativity, HL) and the AQA A-level option 3.12.3 (Turning points: special relativity). Check your own specification for the exact wording.

**Before this lesson:** Lesson 2, *Projectile motion* (adding velocities and changing frame); Lesson 5, *Electric fields and potential* (the energy eV gained across a potential difference). Also assumed: momentum and kinetic energy.

**Leads to:** the end of this route for now.

**You will be able to:**

1. state the two postulates of special relativity and calculate $\gamma = 1/\sqrt{1 - v^2/c^2}$
2. use time dilation $t = \gamma t_0$ and length contraction $L = L_0/\gamma$, identifying proper time and proper length
3. explain the muon experiment from both the Earth's and the muon's point of view
4. add velocities relativistically, and use the Lorentz transformation and the spacetime interval
5. use $E = \gamma m_0c^2$ and $E_k = (\gamma - 1)m_0c^2$, and explain why nothing with mass reaches $c$

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Before some experiments a **Predict** box asks you to commit to an answer; write it down, run the cell, then answer the **Explain** box below it. Answers to both are at the very end, after the worked solutions. Exercises are at the end, with worked solutions below them.

In [ ]:
# Install the course API in a fresh Colab runtime or upgrade an older installation.
import importlib.util
import subprocess
import sys
from importlib.metadata import version

if importlib.util.find_spec("qplay") is None or int(version("quantum-playground").split(".")[0]) < 1:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quantum-playground>=1,<2"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from qplay import relativity

c = 2.998e8           # m/s
e = 1.602e-19         # C
m_e = 9.109e-31       # kg

## 1. Two postulates and the Lorentz factor

Einstein's special relativity (1905) rests on two postulates:

1. The laws of physics are the same in every **inertial** (non-accelerating) frame of reference.
2. The speed of light in a vacuum, $c$, is the same for every inertial observer, however fast they move relative to the source.

The second one is shocking: if you chase a light beam at $0.9c$ it still moves away from you at $c$. Everything else follows from it, and almost every formula involves the **Lorentz factor**

$$\gamma = \frac{1}{\sqrt{1 - v^2/c^2}}.$$

In [ ]:
beta = np.linspace(0, 0.995, 500)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(beta, relativity.gamma(beta * c))
ax1.set(xlabel="v / c", ylabel=r"$\gamma$", title=r"$\gamma$ stays near 1, then shoots up"); ax1.set_ylim(0, 10)
for b in [0.1, 0.5, 0.8, 0.9, 0.99]:
    print(f"v = {b:4.2f}c:  gamma = {relativity.gamma(b * c):.4f}")
small = np.linspace(0, 0.3, 200)
ax2.plot(small, relativity.gamma(small * c) - 1, label=r"$\gamma - 1$")
ax2.plot(small, small**2 / 2, "k--", label=r"$\beta^2/2$")
ax2.set(xlabel="v / c", title=r"At low speed $\gamma \approx 1 + \frac{1}{2}v^2/c^2$"); ax2.legend();

At everyday speeds $\gamma - 1$ is tiny: about $3\times10^{-13}$ for a jet airliner at 250 m s$^{-1}$. That is why Newton's mechanics works so well. Only near $c$ do the effects become large.

## 2. Time dilation

A clock moving relative to you runs **slow**. If a clock records a time $t_0$ between two events that happen at the same place in its own frame (the **proper time**), you measure

$$t = \gamma\,t_0 .$$

**The muon test.** Muons are made by cosmic rays about 15 km up and live for only $2.2\ \mu$s on average (in their own frame). Even at nearly $c$, in $2.2\ \mu$s they would travel only about 660 m, so almost none should reach the ground. Lots do.

**Predict (P1).** Muons moving at $0.995c$ are made 15 km up and live $2.2\ \mu$s on average. What fraction reach the ground: about 1 in 10 billion, about 1 in 10, or nearly all? Write your prediction down before you run the next cell.

In [ ]:
tau, height = 2.2e-6, 15e3          # muon mean lifetime (s) and height of production (m)
speeds = np.linspace(0.9, 0.9999, 400)
t_flight = height / (speeds * c)                       # flight time in the Earth frame
newton = np.exp(-t_flight / tau)                       # if the muon's clock ran at our rate
einstein = np.exp(-t_flight / relativity.time_dilation(tau, speeds * c))
plt.figure(figsize=(7, 3.8))
plt.semilogy(speeds, newton, label="no time dilation")
plt.semilogy(speeds, einstein, label=r"with time dilation, lifetime $\gamma\tau$")
plt.xlabel("muon speed / c"); plt.ylabel("fraction reaching the ground"); plt.legend()
b = 0.995
print(f"at {b}c: gamma = {relativity.gamma(b * c):.2f};  fraction surviving 15 km: "
      f"{np.exp(-height / (b * c) / tau):.1e} without, {np.exp(-height / (b * c) / relativity.time_dilation(tau, b * c)):.3f} with time dilation")

**Explain (P1).** Why do Newton's physics and relativity give such different answers?

Without time dilation about one muon in ten billion would survive; with it, about one in ten does, which is what detectors find.

## 3. Length contraction

The muon's point of view must give the same answer, but in its own frame it is at rest and lives $2.2\ \mu$s. The resolution: in the muon's frame the **atmosphere is contracted**. An object of proper length $L_0$ (measured at rest) is measured as

$$L = \frac{L_0}{\gamma}$$

by an observer it moves past, but only along the direction of motion.

In [ ]:
b = 0.995
L_muon = relativity.length_contraction(height, b * c)
print(f"In the muon's frame the 15 km of atmosphere is {L_muon / 1e3:.2f} km thick.")
print(f"Time for the ground to reach the muon at {b}c: {L_muon / (b * c) * 1e6:.2f} microseconds "
      f"= {L_muon / (b * c) / tau:.2f} mean lifetimes.")
print(f"Earth frame: flight takes {height / (b * c) * 1e6:.1f} us = {height / (b * c) / relativity.time_dilation(tau, b * c):.2f} dilated lifetimes. Same answer.")

Both observers agree on the number of lifetimes the trip takes, and so on how many muons arrive. They disagree about *why*: the Earth observer says the muon's clock runs slow; the muon says the atmosphere is short.

## 4. Adding velocities

If a spaceship moving at $v$ fires a probe forwards at $u$ (relative to the ship), Newton would say the probe moves at $u + v$. Relativity says

$$w = \frac{u + v}{1 + uv/c^2},$$

which is never more than $c$. Fire a light pulse ($u = c$) and you get $w = c$ for any $v$: the second postulate.

In [ ]:
v = np.linspace(0, 1, 200)
plt.figure(figsize=(6.5, 4))
for u in [0.3, 0.6, 0.9, 1.0]:
    plt.plot(v, relativity.add_velocities(u * c, v * c) / c, label=f"probe at {u}c relative to ship")
    plt.plot(v, u + v, ":", color=plt.gca().lines[-1].get_color(), lw=0.8)
plt.axhline(1, color="k", lw=0.8); plt.ylim(0, 2)
plt.xlabel("ship speed / c"); plt.ylabel("probe speed seen from Earth / c"); plt.legend(fontsize=8)
plt.title("Solid: relativity.  Dotted: Newton's u + v");

## 5. Spacetime: what everyone agrees on

The **Lorentz transformation** converts the time and position of an event between frames moving at relative speed $v$ along $x$:

$$ct' = \gamma\,(ct - \beta x),\qquad x' = \gamma\,(x - \beta\,ct),\qquad \beta = v/c.$$

Observers disagree about times and lengths, but they all agree on the **spacetime interval** between two events,

$$s^2 = -(c\Delta t)^2 + \Delta x^2 + \Delta y^2 + \Delta z^2 .$$

Below, two lightning strikes are simultaneous in the station frame. Seen from a train moving at $0.6c$ they are not, but the interval is unchanged.

**Predict (P2).** Lightning strikes A (behind) and B (ahead) are simultaneous for someone on the platform. For a passenger on a train moving towards B at $0.6c$, which strike happens first, or are they still simultaneous? Write your prediction down before you run the next cell.

In [ ]:
L = 600.0                                     # the strikes are 600 m apart
A = np.array([0, -L / 2, 0, 0])               # (ct, x, y, z) in metres, station frame
B = np.array([0, +L / 2, 0, 0])
boost = relativity.lorentz_boost(0.6)
A_train, B_train = boost @ A, boost @ B
print(f"station: strikes at t = {A[0] / c * 1e6:.3f} us and {B[0] / c * 1e6:.3f} us (simultaneous)")
print(f"train:   strikes at t = {A_train[0] / c * 1e6:.3f} us and {B_train[0] / c * 1e6:.3f} us (B first!)")
print(f"interval s^2: station {relativity.interval(B - A):.1f} m^2, train {relativity.interval(B_train - A_train):.1f} m^2")

**Explain (P2).** Which quantity did both observers agree on, and why does that matter?

**Simultaneity is relative.** There is no universal "now". But cause and effect are safe: if one event can cause another ($s^2 < 0$, "timelike"), every observer agrees on their order.

## 6. Mass and energy

The total energy of a particle of rest mass $m_0$ is $E = \gamma m_0c^2$, so even at rest it has $E_0 = m_0c^2$. Its kinetic energy is

$$E_k = (\gamma - 1)\,m_0c^2,$$

which approaches $\tfrac12 m_0v^2$ at low speed but grows without limit as $v \to c$: no amount of energy gets a massive particle to the speed of light. Below, electrons are accelerated through a potential difference $V$, gaining $eV$ of kinetic energy.

**Predict (P3).** Newton's $\tfrac12mv^2 = eV$ says an electron accelerated through 1 MV reaches about $2c$. What does relativity give: just under $c$, exactly $c$, or about $2c$? Write your prediction down before you run the next cell.

In [ ]:
V = np.logspace(2, 7, 300)                     # 100 V to 10 MV
gam = 1 + e * V / (m_e * c**2)
beta_rel = np.sqrt(1 - 1 / gam**2)
beta_newton = np.sqrt(2 * e * V / m_e) / c
plt.figure(figsize=(7, 3.8))
plt.semilogx(V, beta_newton, "--", label=r"Newton: $\frac{1}{2}mv^2 = eV$")
plt.semilogx(V, beta_rel, label=r"relativity: $(\gamma - 1)mc^2 = eV$")
plt.axhline(1, color="k", lw=0.8); plt.ylim(0, 2.5)
plt.xlabel("accelerating p.d. / V"); plt.ylabel("electron speed / c"); plt.legend()
print(f"electron rest energy m c^2 = {m_e * c**2 / e / 1e6:.3f} MeV")

**Explain (P3).** Where does the extra energy go, if not into speed?

Above about 100 kV Newton's formula goes badly wrong and soon predicts speeds faster than light. This was tested directly by Bertozzi in 1964: electrons from a 15 MeV accelerator reached $0.9995c$, not the $7.7c$ Newton predicts.

The `relativity.kinetic_energy` function computes $(\gamma - 1)m_0c^2$ using $\gamma - 1 = \beta^2\gamma^2/(\gamma + 1)$, which avoids subtracting two nearly equal numbers at low speed.

---

## Exercises

Take $c = 3.00\times10^{8}\ \mathrm{m\,s^{-1}}$. Give answers to 3 significant figures.

**Q1.** Find $\gamma$ for a speed of $0.80c$.

**Q2.** A spaceship travels at $0.60c$ to a star. The trip takes 4.0 years as measured on Earth. How long does it take according to the crew's clocks?

**Q3.** Muons have a mean lifetime of $2.2\ \mu$s in their rest frame. For muons moving at $0.98c$, find (a) their mean lifetime measured in the Earth frame, (b) the distance they travel in that time, in km.

**Q4.** Two spaceships fly towards each other, one at $0.70c$ and the other at $0.60c$ relative to Earth. How fast does one move relative to the other, as a fraction of $c$?

**Q5.** An electron (rest energy 0.511 MeV) is accelerated from rest through 1.0 MV. Find its speed as a fraction of $c$.

**Q6.** An event happens 5.0 years later and 3.0 light-years away from you. What is the proper time between you now and that event, for a traveller who goes from one to the other at constant velocity? (Use the interval.)

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_g5 = 1 + 1.0e6 / 0.511e6                    # gamma from (gamma - 1) m c^2 = eV, energies in eV
_event = np.array([5.0, 3.0, 0, 0])          # (ct, x, y, z) in light-years
_answers = {
    "Q1": float(relativity.gamma(0.80, c=1.0)),
    "Q2": 4.0 / relativity.gamma(0.60, c=1.0),
    "Q3a": relativity.time_dilation(2.2e-6, 0.98, c=1.0),
    "Q3b": 0.98 * c * relativity.time_dilation(2.2e-6, 0.98, c=1.0) / 1e3,
    "Q4": relativity.add_velocities(0.70, 0.60, c=1.0),
    "Q5": np.sqrt(1 - 1 / _g5**2),
    "Q6": np.sqrt(-relativity.interval(_event)),
}

# Replace None with your answers (Q2 in years, Q3a in s, Q3b in km, Q4 and Q5 as fractions of c, Q6 in years).
mine = {"Q1": None, "Q2": None, "Q3a": None, "Q3b": None, "Q4": None, "Q5": None, "Q6": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** $\gamma = 1/\sqrt{1 - 0.80^2} = 1/\sqrt{0.36} = 1/0.6 = 1.67$

**Q2.** The crew's clocks are present at both departure and arrival, so they measure the proper time: $t_0 = t/\gamma = 4.0 \times \sqrt{1 - 0.6^2} = 4.0 \times 0.8 = 3.20$ years.

**Q3.** $\gamma = 1/\sqrt{1 - 0.98^2} = 5.03$. (a) $t = \gamma t_0 = 5.03 \times 2.2\ \mu\mathrm s = 1.11\times10^{-5}$ s &nbsp; (b) $d = vt = 0.98 \times 3.00\times10^8 \times 1.106\times10^{-5} = 3250$ m $= 3.25$ km, compared with 0.65 km without time dilation.

**Q4.** In the frame of the first ship, Earth moves at $0.70c$ and the other ship comes at $0.60c$ relative to Earth, so $w = \dfrac{0.70 + 0.60}{1 + 0.70 \times 0.60}c = \dfrac{1.30}{1.42}c = 0.915c$. Newton would say $1.30c$.

**Q5.** $E_k = 1.0$ MeV, so $(\gamma - 1) \times 0.511 = 1.0$ and $\gamma = 2.957$. Then $\beta = \sqrt{1 - 1/\gamma^2} = 0.941$, so $v = 0.941c$. (Newton's $\tfrac12mv^2 = eV$ would give $1.98c$.)

**Q6.** In units of years and light-years ($c = 1$), $s^2 = -5.0^2 + 3.0^2 = -16$. For a timelike interval the proper time is $\tau = \sqrt{-s^2} = 4.00$ years. The traveller moves at $0.6c$, and indeed $5.0/\gamma = 5.0 \times 0.8 = 4.0$.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}")
# Q6 again: boost into the traveller's frame (beta = 3/5) and the event happens at x' = 0 after 4 years.
print("event in the traveller's frame (ct', x'):", np.round(relativity.lorentz_boost(0.6) @ _event, 12)[:2])

**Next:** in section 5, change the train's speed. At what speed is the time between the two strikes, seen from the train, equal to 1 microsecond?

### Predict and explain: answers

**P1.** About 1 in 10 (0.103 in the printout). The trip takes $50\ \mu$s in the Earth's frame: about 23 lifetimes if the muon's clock ran at our rate, leaving $e^{-23} \approx 10^{-10}$. But its clock runs slow by $\gamma = 10$, so the trip lasts only 2.3 of its own lifetimes, and $e^{-2.3} \approx 0.1$ survive.

**P2.** B happens first, about 1.5 µs before A for the passenger. Whether two separated events are simultaneous depends on the frame. Both observers agree on the spacetime interval, $s^2 = 360\,000\ \mathrm{m^2}$: it is the same in every inertial frame, which is what makes it physically meaningful.

**P3.** Just under $c$: $0.94c$ (exercise Q5). As $v$ approaches $c$, $\gamma$ grows without limit, so each extra joule of kinetic energy buys less and less speed. The energy goes into the particle's total energy $\gamma m_0c^2$, which keeps rising while the speed creeps towards $c$.